<a href="https://colab.research.google.com/github/danieligelnik/CCFraudProject/blob/main/help_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays as hol
from sklearn.model_selection import train_test_split, KFold

In [ ]:
def load_kagglehub_dataset(dataset_path, file_name):
  # Load the latest version
  return kagglehub.dataset_load(
      KaggleDatasetAdapter.PANDAS,
      dataset_path,
      file_name,
      # Provide any additional arguments like
      # sql_query or pandas_kwargs. See the
      # documenation for more information:
      # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
      )

In [ ]:
def df_info(df, what):
  if (what == "head"):
    print("First 5 records:")
    with pd.option_context('display.max_columns', 50):
      display(df.head())
  elif (what == "columns"):
    print("Columns:")
    display(df.columns)
    print(f"Rows / columns: {df.shape}")
  elif (what == "info"):
    print("Missing values:")
    display(df.isna().sum())
    display(df.info())

In [ ]:
from sklearn.feature_extraction import FeatureHasher

# Feature extraction using FeatureHasher on high-cardinality columns for given data set
def extract_features(df):
  hasher = FeatureHasher(n_features=10, input_type='string')
  # Example: Extracting features from 'merchant' and 'job' column for the training set
  # Note: This is applied only to df_cards_train
  merchant_features = hasher.transform(df['merchant'].astype(str).apply(lambda x: [x]))
  job_features = hasher.transform(df['job'].astype(str).apply(lambda x: [x]))

  # Convert to arrays and join back to the training dataframe
  merchant_df = pd.DataFrame(merchant_features.toarray(), columns=[f'merch_feat_{i}' for i in range(10)], index=df.index)
  job_df = pd.DataFrame(job_features.toarray(), columns=[f'job_feat_{i}' for i in range(10)], index=df.index)

  df = pd.concat([df, merchant_df, job_df], axis=1)

  return df

In [ ]:
def feature_engineering(df):
  df_new['gender'] = df['gender'].map({'F': 1, 'M': 0})
  df_new = pd.get_dummies(df_new, columns=['category'], prefix='category', drop_first=True, dtype=int)
  df_new['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
  df_new['age'] = (df['trans_date_trans_time'] - pd.to_datetime(df.dob)).dt.days
  df_new['is_weekend'] = df['trans_date_trans_time'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
  df_new['trans_date'] = df['trans_date_trans_time'].dt.date.apply(lambda x: x.toordinal()).astype(np.uint64)
  df_new = pd.get_dummies(df, columns=['category'], prefix='category', drop_first=True, dtype=int)

  df_new = extract_features(df_new)

  return df_new
